# 09 - Model Baseline: Regresi Logistik

## Context
Sebelum menggunakan algoritma gradient boosting yang kompleks, kita perlu membuat model baseline sederhana menggunakan Logistic Regression. Model ini berfungsi sebagai pembanding performa dasar, membantu kita memahami performa awal, dan melihat fitur mana saja yang berpengaruh secara linear.

## Objective
- Melatih model Logistic Regression dengan preprocessing yang sesuai (imputasi dan penskalaan).
- Mengevaluasi performa model menggunakan metrik ROC-AUC dan Precision-Recall.
- Menganalisis tingkat pengaruh fitur berdasarkan koefisien regresi.
- Menetapkan batas performa minimum (performance floor) untuk evaluasi model yang lebih kompleks.

### Impor Library & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import time

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    confusion_matrix, classification_report
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Load data hasil feature engineering
data_path = Path('../data/processed/train_features.parquet')
if not data_path.exists():
    data_path = Path('../data/interim/train_merged.parquet')

train = pd.read_parquet(data_path)
print(f'Data loaded: {train.shape}')

### Persiapan Fitur & Target

In [ ]:
exclude_cols = ['TransactionID', 'isFraud', 'TransactionDT']

# Regresi Logistik membutuhkan fitur numerik saja
numeric_cols = train.select_dtypes(include=['int64', 'float64', 'int32', 'float32']).columns.tolist()
feature_cols = [c for c in numeric_cols if c not in exclude_cols]

X = train[feature_cols]
y = train['isFraud']

### Split Data dengan Stratifikasi

In [ ]:
# Bagi data ke train dan validation set dengan stratifikasi target
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train set shape: {X_train.shape}, positive class: {y_train.sum()} ({y_train.mean()*100:.2f}%)")
print(f"Validation set shape: {X_val.shape}, positive class: {y_val.sum()} ({y_val.mean()*100:.2f}%)")

### Pelatihan Model Baseline (Logistic Regression)

In [ ]:
# Buat pipeline preprocessing + modeling
# Gunakan class_weight='balanced' untuk menangani kelas yang tidak seimbang
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42, verbose=1))
])

print("Training baseline Logistic Regression model...")
start_time = time.time()
pipeline.fit(X_train, y_train)
print(f"Training completed in {time.time() - start_time:.2f} seconds.")

### Evaluasi Model Baseline

In [ ]:
# Hitung probabilitas prediksi
y_train_proba = pipeline.predict_proba(X_train)[:, 1]
y_val_proba = pipeline.predict_proba(X_val)[:, 1]

# Hitung skor AUC-ROC
train_auc = roc_auc_score(y_train, y_train_proba)
val_auc = roc_auc_score(y_val, y_val_proba)

print(f"Train ROC-AUC: {train_auc:.4f}")
print(f"Validation ROC-AUC: {val_auc:.4f}")
print(f"Overfitting Gap: {train_auc - val_auc:.4f}")

# Classification report
y_val_pred = pipeline.predict(X_val)
print("\nClassification Report (Default Threshold 0.5):")
print(classification_report(y_val, y_val_pred))

print("Confusion Matrix (Default Threshold 0.5):")
print(confusion_matrix(y_val, y_val_pred))

### Visualisasi: Kurva ROC & Kurva Precision-Recall

In [ ]:
fpr, tpr, _ = roc_curve(y_val, y_val_proba)
roc_auc = auc(fpr, tpr)

precision, recall, _ = precision_recall_curve(y_val, y_val_proba)
ap = average_precision_score(y_val, y_val_proba)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot Kurva ROC
axes[0].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.4f})')
axes[0].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[0].set_xlim([0.0, 1.0])
axes[0].set_ylim([0.0, 1.05])
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('Receiver Operating Characteristic (ROC)')
axes[0].legend(loc="lower right")

# Plot Kurva Precision-Recall
axes[1].plot(recall, precision, color='blue', lw=2, label=f'PR curve (AP = {ap:.4f})')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend(loc="lower left")

plt.tight_layout()
plt.show()

### Analisis Koefisien Model (Feature Importance)

In [ ]:
model = pipeline.named_steps['model']
coefs = model.coef_[0]

coef_df = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': coefs,
    'abs_coefficient': np.abs(coefs)
}).sort_values(by='abs_coefficient', ascending=False)

print("Top 15 Most Influential Features (Logistic Regression Coefficients):")
print(coef_df.head(15))

# Plot koefisien
plt.figure(figsize=(10, 6))
top_coefs = coef_df.head(15).sort_values(by='coefficient')
colors = ['red' if c < 0 else 'green' for c in top_coefs['coefficient']]
plt.barh(top_coefs['feature'], top_coefs['coefficient'], color=colors)
plt.title('Top 15 Features by Logistic Regression Coefficient')
plt.xlabel('Coefficient Value')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()